# Error analysis

**Group 4**


`Melina Paxinou    2854344`

## Imports

In [5]:
import pandas as pd
import random
import csv
from operator import itemgetter
import string
from types import GeneratorType
import networkx as nx
import pickle
import benepar
import sklearn_crfsuite
import spacy
from spacy.tokens import Doc
from spacy import displacy
from nltk.tree import Tree
from collections import Counter
import en_core_web_md
from spacy.tokenizer import Tokenizer
from spacy.util import compile_infix_regex

In [6]:
nlp = en_core_web_md.load()
nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

C:\Users\melou\anaconda3\Lib\site-packages\benepar\parse_chart.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(
You are using the default legac

## Predictions file

This code is a demonstration of how I created the combined test and development predictions file.

In [9]:
test_pred = 'Test_predictions.txt'
dev_pred = 'dev_pred.txt'

In [10]:
combined_dev_test_pred = 'combined_dev_test_pred.txt'

# combining dev and test prediction files
with open(combined_dev_test_pred, 'w') as new_file:
    with open(test_pred, 'r') as file1:
        for line in file1:
            if line.strip():
                new_file.write(line)

    with open(dev_pred, 'r') as file2:
        for line in file2:
            if line.strip():
                new_file.write(line)

## Error patterns

This is the code that was used to create the .csv file that contains the 50 sentences which contain errors. The sentences were randomly selected from the file which contained the information and predictions of the development and test sets. 

In [13]:
# loading the dataset
columns = [
    "chapter_name",
    "sentence_number",
    "token_number",
    "word",
    "lemma",
    "part-of-speech",
    "syntax",
    "cue",
    "scope",
    "event",
    "pred",
]

df = pd.read_csv(combined_dev_test_pred, sep="\t", header=0, names=columns)

# initializing error type column
df["error_type"] = None

# function that assigns FPs and FNs according to the content of pred and scope columns
def assign_error_type(row):
    if row["pred"] != "_" and row["scope"] == "_":
        return "FP"
    elif row["pred"] == "_" and row["scope"] != "_":
        return "FN"
    return None

# applying the function to each row in the dataframe (specified by axis=1) and saving to error_type column
df["error_type"] = df.apply(assign_error_type, axis=1)

# grouping by chapter_name and sentence_number
sentences = list(df.groupby(["chapter_name", "sentence_number"], sort=False))

# shuffling the sentences so that they are taken from different places in the file
random.shuffle(sentences)

# FP and FN lists and set for sentences that have been checked
fp_samples = []
fn_samples = []
used_sentences = set()

# sentences with FP
for (chapter_name, sentence_number), group in sentences:
    if len(fp_samples) >= 25:
        break
    if any(group["error_type"] == "FP") and (chapter_name, sentence_number) not in used_sentences:
        fp_samples.append(group)
        used_sentences.add((chapter_name, sentence_number))

# sentences with FN 
for (chapter_name, sentence_number), group in sentences:
    if len(fn_samples) >= 25:
        break
    if any(group["error_type"] == "FN") and (chapter_name, sentence_number) not in used_sentences:
        fn_samples.append(group)
        used_sentences.add((chapter_name, sentence_number))

# combining FP and FN samples
error_samples = pd.concat(fp_samples + fn_samples)

Please note that the following cell is commented so that the randomly selected instances in the error_samples.csv file are not overwritten. The .csv file is in my submission folder.

In [15]:
# saving to csv
# ATTENTION: if you uncomment the following line, then the csv file that is saved will be overwritten and the error categories will disappear
# error_samples.to_csv("error_samples.csv", sep=",", index=False)

In this part of the notebook, the .csv file is loaded and I am printing the sentences for convenience purposes, and the relevant df columns that will help determine the error categories.

In [17]:
errors = 'error_samples.csv'

columns = [
    "chapter_name",
    "sentence_number",
    "token_number",
    "word",
    "lemma",
    "part-of-speech",
    "syntax",
    "cue",
    "scope",
    "event",
    "pred",
    "error_type",
    "error_category"
]

error_df = pd.read_csv(errors, sep=",", header=0, names=columns)

error_sentences = list(error_df.groupby(["chapter_name", "sentence_number"], sort=False))

In [18]:
for (chapter, sentence_num), group in error_sentences:
    # joining the words to form a sentence
    sentence = ' '.join(group['word'].values)
    print(f"Chapter: {chapter}, Sentence: {sentence_num}")
    print(f"Sentence: {sentence}")
    print("\n")

Chapter: circle01, Sentence: 167_1
Sentence: The mysterious one could understand English , even if he could not print it .


Chapter: circle01, Sentence: 140_1
Sentence: I can imagine that the word was taken out of a dictionary , which would give the noun but not the plural .


Chapter: circle01, Sentence: 294_1
Sentence: I suppose when you doctored you found yourself studying cases without thought of a fee ? ''


Chapter: wisteria02, Sentence: 164_2
Sentence: `` I did n't say so , Mr. Holmes ; I did n't say so .


Chapter: cardboard, Sentence: 108_1
Sentence: `` What is the use of asking me questions when I tell you I know nothing whatever about it ? ''


Chapter: cardboard, Sentence: 225_1
Sentence: `` I wonder , since you are both maiden ladies , that you do not keep house together . ''


Chapter: wisteria02, Sentence: 144_1
Sentence: Pray do n't think it a liberty if I give you a word of friendly warning . ''


Chapter: cardboard, Sentence: 23_1
Sentence: On my remarking that I was

In [19]:
print(f"{'Chapter Name':<15} {'Sentence Number':<15} {'Word':<15} {'Cue':<15} {'Scope':<15} {'Pred':<15} {'Error Type':<15}")
print('------------------------------------------------------------------------------------------------------------')
for _, group in error_sentences:  
    for index, row in group.iterrows():
        print(f"{row['chapter_name']:<15} {row['sentence_number']:<15} {row['word']:<15} {row['cue']:<15} {row['scope']:<15} {row['pred']:<15} {row['error_type']:<15}")
    print('------------------------------------------------------------------------------------------------------------')

Chapter Name    Sentence Number Word            Cue             Scope           Pred            Error Type     
------------------------------------------------------------------------------------------------------------
circle01        167_1           The             _               The             The             _              
circle01        167_1           mysterious      _               mysterious      mysterious      _              
circle01        167_1           one             _               one             one             _              
circle01        167_1           could           _               _               could           FP             
circle01        167_1           understand      _               _               understand      FP             
circle01        167_1           English         _               _               _               _              
circle01        167_1           ,               _               _               _               _          

In [20]:
error_category_count = {}

# iterating over the grouped sentences
for group, group_df in error_sentences:
    # checking the error categories for each group
    error_categories_in_sentence = group_df['error_category'].dropna().unique()
    
    for error_category in error_categories_in_sentence:
        if error_category != '':
            # splitting categories separated by ' + ' and counting them separately
            individual_categories = error_category.split(' + ')
            
            for category in individual_categories:
                if category not in error_category_count:
                    error_category_count[category] = 0
                error_category_count[category] += 1

# sorting the error categories by count in descending order using itemgetter
sorted_error_categories = sorted(error_category_count.items(), key=itemgetter(1), reverse=True)

# printing the sentence counts per error category, sorted by frequency
for error_category, count in sorted_error_categories:
    print(f"Error Category: {error_category} - Sentences Count: {count}")

Error Category: MC_and_SC - Sentences Count: 6
Error Category: Sub_CL_only - Sentences Count: 5
Error Category: No_pattern - Sentences Count: 5
Error Category: Pred_exp_OOS - Sentences Count: 5
Error Category: Aff_skipped - Sentences Count: 4
Error Category: Wrong_subj - Sentences Count: 3
Error Category: Adj_to_clause - Sentences Count: 3
Error Category: PP_skipped - Sentences Count: 3
Error Category: SC_not_MC - Sentences Count: 2
Error Category: Repeat_OOS - Sentences Count: 2
Error Category: Int_NIS - Sentences Count: 2
Error Category: PP_OOS - Sentences Count: 2
Error Category: Diff_pred - Sentences Count: 2
Error Category: Prep_to_C - Sentences Count: 1
Error Category: MC_not_SC - Sentences Count: 1
Error Category: Adv_OOS - Sentences Count: 1
Error Category: NP_only - Sentences Count: 1
Error Category: MN_WC - Sentences Count: 1
Error Category: Other_CL - Sentences Count: 1
Error Category: Adv_v - Sentences Count: 1
Error Category: Pro_aff_miss - Sentences Count: 1


## Features

This section is a copy of assignment 2 features, as they help determine important information for the error analysis, such as dependencies. I have added the lemma to the list of features. Please note that there is no need to run this part, as in my submission you will find the pickle files used to perform the counterfactual error analysis.

In [23]:
# inspired by: https://towardsdatascience.com/how-to-find-shortest-dependency-path-with-spacy-and-stanfordnlp-539d45d28239
# and https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html

def find_dep_path(sent, word, negation):
    """
    Finds the dependency path starting from each token of a sentence and ending with the negation cue.
    If the negation cue is more than one word, then it finds the dependency path between the token 
    and the first token of the multi negation cue. It uses networkx's "shortest_path" algorithm to identify
    the shortest path between the cue and each word (list of tokens) and then that path is converted into 
    a list of dependencies. 

    Parameters:
    sent: (list) a list of words that make a sentence
    word: (str) individual word of the sentence
    negation: (str) a negation cue 

    Returns the dependency path in a list, or, if not applicable, it returns 'No Path'.
    """
    spacy_doc = Doc(nlp.vocab, words=sent)
    processed_doc = nlp(spacy_doc)

    if not (sent and word and negation):
        return ['No Path']
    edges = []

    for token in processed_doc:
        for child in token.children:
            edges.append(('{0}'.format(token.lower_),
                          '{0}'.format(child.lower_)))
    graph = nx.Graph(edges)
    entity1 = str(word).lower()
    entity2 = str(negation).lower()

    try:
        dep_path = nx.shortest_path(graph, source=entity1, target=entity2)

        may_dep_path = []
        for i in dep_path:
            index = [token.lower() for token in sent].index(i.lower())
            a = processed_doc[index].dep_
            may_dep_path.append(a)

        return may_dep_path

    except nx.NetworkXNoPath:
        # Handle cases where no path exists
        return ["No Path"]


# inspired by the paper "A Conditional Random Field Model for Resolving the Scope of Negation" by Amjad Abu-Jbara and Dragomir Radev

def find_neg_type(row, neg_group):
    """
    Finds the type of negation cue in each sentence. The possible negations are: 
    multi: multi-word negation cues
    neg: single-word negation cues
    pre: prefix negation cues
    post: postfix negation cues
    It assigns a negation type per sentence. 

    Parameters: 
    row: the row of a pandas dataframe, which contains a 'word' and a 'cue' column 
    neg_group: a list of negation cue tokens in a sentence

    Returns the negation type (str). 
    """
    if len(neg_group) > 1:
        neg_type = 'multi'
    elif row['word'] == row['cue']:
        neg_type = 'neg'
    else:
        neg_type = 'pre' if row['word'].startswith(row['cue']) else 'post'

    return neg_type

# Bidirectional dependency distance: https://aclanthology.org/W15-2914.pdf
# Dependency relation: https://pmc.ncbi.nlm.nih.gov/articles/PMC3392064/


def find_dependency(word, processed_doc):
    """
    Finds the dependency of a word in a sentence. 

    Parameters: 
    word: (str) individual word of the sentence
    processed_doc: sentence processed by spacy

    Returns the dependency of the word. 
    """
    for i in processed_doc:
        if (word == i.text):
            return i.dep_


def find_lemma(word, processed_doc):
    """
    Finds the dependency of a word in a sentence. 

    Parameters: 
    word: (str) individual word of the sentence
    processed_doc: sentence processed by spacy

    Returns the dependency of the word. 
    """
    for i in processed_doc:
        if (word == i.text):
            return i.lemma_

In [24]:

def find_paths_to_words(tree, target_words, current_path=None):
    """
    Find paths to the target words in the tree.

    Parameters:
    - tree: NLTK Tree object or string
    - target_words: List of words to find in the tree
    - current_path: Current path in the tree (used in recursion)

    Returns:
    - dict: Dictionary mapping found words to their paths
    """

    if current_path is None:
        current_path = []

    paths = {}

    # Skip the tree if it's a generator
    if isinstance(tree, GeneratorType):
        return paths  # Return an empty dictionary

    # If it's a leaf node (word) and it matches a target word, record its path.
    if isinstance(tree, str):
        if tree in target_words:
            paths[tree] = current_path
        return paths

    # If it's a Tree node
    if isinstance(tree, Tree):
        for i, child in enumerate(tree):

            # why tree.label() not child.label()? -- each node's path should record its parent's label, not its own label
            # (tree.label(), i) means "we went through a node of type X and took its i-th branch"

            new_path = current_path + [(tree.label(), i)]
            child_paths = find_paths_to_words(child, target_words, new_path)
            paths.update(child_paths)

    return paths


def find_common_ancestor(parse_tree, word1, word2):
    """
    Find the lowest common ancestor node type between two words in an NLTK parse tree.

    Parameters:
    - parse_tree (nltk.Tree): The parse tree to search
    - word1 (str): First word to find
    - word2 (str): Second word to find

    Returns:
    - str: The node type of the lowest common ancestor, or None if words are not found
    """
    # Find paths to the words
    word_paths = find_paths_to_words(parse_tree, [word1, word2])

    # Check if both words were found
    if len(word_paths) < 2:
        return []

    # Find the lowest common ancestor
    path1 = word_paths[word1]
    path2 = word_paths[word2]

    # Find the common prefix of the paths
    common_ancestor = ''
    for (node1, index1), (node2, index2) in zip(path1, path2):
        if node1 == node2:
            common_ancestor = node1
        else:
            break

    return common_ancestor

def parse_sent(sentence):
    """
    Helper function to parse sentence and extract POS features including dependency information
    Args:
        sentence: String or list of tokens
    Returns:
        pos_features: List of dictionaries containing POS features for each token
    """
    if isinstance(sentence, list):
        spacy_doc = Doc(nlp.vocab, words=sentence)
        processed_doc = nlp(spacy_doc)
    else:
        processed_doc = nlp(sentence)

    pos_features = []

    for sent in processed_doc.sents:
        tree = sent._.parse_string
        for token in sent:
            # Basic POS features
            token_pos = token.pos_
            dep1_pos = token.head.pos_ if token.head != token else "ROOT"
            dep2_pos = token.head.head.pos_ if token.head.head != token.head else "ROOT"

            pos_features.append({
                "pos": token_pos,
                "dep1_pos": dep1_pos,
                "dep2_pos": dep2_pos
            })

    return tree, pos_features, processed_doc


def extract_features(sentence_data):
    """
    Extract features from a single sentence
    Args:
        sentence_data: DataFrame containing the sentence with all columns
    Returns:
        List of dictionaries containing features for each token
    """
    # Get words and cues
    # a pandas Series --> a Python list
    words = sentence_data['word'].tolist()
    cues = sentence_data['cue'].tolist()

    # Parse sentence
    tree, pos_features, processed_doc = parse_sent(words)

    # Convert parse string to NLTK Tree
    if isinstance(tree, str):
        tree = Tree.fromstring(tree)

    # Find negation cues
    neg_cues_idx = []
    neg_tokens = []  # spaCy token
    neg_words = []  # Store the actual words

    for idx, (word, cue) in enumerate(zip(words, cues)):
        is_neg = not (cue == "***" or cue == "_")
        if is_neg:
            neg_cues_idx.append(idx)
            neg_tokens.append(list(processed_doc)[idx])
            neg_words.append(word)
    row = sentence_data[sentence_data['cue'] != '_'].iloc[0]
    neg_type = find_neg_type(row, neg_tokens)

    # Process each token
    sentence_features = []
    for idx, (word, cue) in enumerate(zip(words, cues)):
        current_token = list(processed_doc)[idx]
        is_neg = idx in neg_cues_idx  # the output will be True/False

        # Find common_ancestor
        common_ancestor = ""
        if not is_neg and len(neg_words) > 0:
            for neg_word in neg_words:
                ancestor = find_common_ancestor(tree, word, neg_word)
                if ancestor != "":
                    common_ancestor = ancestor
                    break  # Take the first valid ancestor we find

        # Token Distance
        distance = 0
        for neg_idx in neg_cues_idx:
            distance = abs(idx - neg_idx)

        # Same Chunk
        same_chunk = False
        token_to_check = " "

        # update token_to_check from non-cue tokens
        if current_token not in neg_tokens:
            token_to_check = current_token.text

            # loop through each negation cue in the sentence and get the path of each cue, paired with path of each token_to_check
            for neg_idx in neg_cues_idx:
                if cues[neg_idx] == words[neg_idx]:
                    paths = find_paths_to_words(
                        tree, [token_to_check, cues[neg_idx]])
                    # check if both tokens exist in paths
                    if token_to_check not in paths or cues[neg_idx] not in paths:
                        continue
                    path1 = paths[token_to_check]
                    path2 = paths[cues[neg_idx]]
                else:
                    paths = find_paths_to_words(
                        tree, [token_to_check, words[neg_idx]])

                    # check if both tokens exist in paths
                    if token_to_check not in paths or words[neg_idx] not in paths:
                        continue
                    path1 = paths[token_to_check]
                    path2 = paths[words[neg_idx]]

                # compare and find the minimum path length among the two
                min_len = min(len(path1), len(path2))

                # check if the common node is a phrase node (NP, VP, PP, etc.), if yes, they're in the same chunk
                common_node = None
                for i in range(min_len-1):
                    if path1[i][0] == path2[i][0] and i > 0 and path1[:i] == path2[:i]:
                        common_node = path1[i][0]  # Update the common node

                # check if the common node is a phrase-level node
                if common_node is not None and common_node not in ['S', 'ROOT']:
                    same_chunk = True
                else:
                    same_chunk = False
        else:
            same_chunk = True

        # Create feature dictionary
        feature_dict = {
                    # base
                    "token": word,
                    "lemma": find_lemma(word, processed_doc),
                    "is_neg": is_neg,
                    # Carol
                    "pos": pos_features[idx]["pos"],
                    "dep1_pos": pos_features[idx]["dep1_pos"],
                    "dep2_pos": pos_features[idx]["dep2_pos"],
                    "common ancestor": common_ancestor,
                    # Melina
                    "dep_path": " ".join(find_dep_path(words, word, neg_tokens[0] if neg_tokens else False)),
                    "neg_type": neg_type,
                    # Matt
                    "dep_path_length": len(find_dep_path(words, word, neg_tokens[0] if neg_tokens else False)),
                    "dep": find_dependency(word, processed_doc),
                    # Shutao
                    "token_distance": distance,
                    "same_chunk": same_chunk
                }

        sentence_features.append(feature_dict)

    return sentence_features

In [25]:
def prepare_data(input_file):
    """
    Prepares data by processing a tab-separated file.

    This function reads a CoNLL-style TSV file and transforms it into a
    suitable format. It creates tokenized sentences with their features
    and also generates target labels.

    Parameters:
    ----------
    input_file : str
        Path to the input TSV file. The file should have the following columns:
        - chapter_name: Name of the chapter or document source.
        - sentence_number: Sentence index within the chapter.
        - token_number: Token index within the sentence.
        - word: The actual token.
        - lemma: Lemmatized form of the word.
        - part-of-speech: POS tag of the token.
        - syntax: Syntactic information.
        - cue: Negation cue indicator ('***' or '_' for no cue and not
                currently the cue, otherwise the word itself).
        - scope: Negation scope annotation.
        - event: Event annotation.

    Returns:
    -------
    train_sentences : list of list of dict
        A list where each element represents a sentence, composed of
        dictionaries of features where every dictionary represents a word.

    target : list of list of str
        A list of target labels for each token in each sentence:
        - "0" if the token is out of scope.
        - "1" if the token is in scope.
    """

    column_names = [
        "chapter_name",
        "sentence_number",
        "token_number",
        "word",
        "lemma",
        "part-of-speech",
        "syntax",
        "cue",
        "scope",
        "event",
        "pred"
    ]

    df = pd.read_csv(
        input_file,
        sep="\t",
        names=column_names,
        encoding="utf-8",
        na_filter=False,
    )

    # double checking to make sure we are not including any sentences that do not include a negation cue
    df = df[df['cue'] != '***']

    sentences = df.groupby(["chapter_name", "sentence_number"], sort=False)

    train_sentences = []
    target = []
    start = False
    for (chapter_name, sentence_number), group in sentences:
        # Tracks progress of chapter and sentence number while processing, comment to remove
        print(f"\rProgress:{chapter_name}-{sentence_number}", end="")
        each_target = []
        for word in group.iterrows():
            if word[1]['scope'] == '_' or word[1]['scope'] == '***' or word[1]['scope'] == '':
                each_target.append('0')
            else:
                each_target.append('1')
        sentence = " ".join(group["word"].tolist())
        each_sentence = extract_features(group)
        train_sentences.append(each_sentence)
        target.append(each_target)
        # if sentence_number == '16_1':
        #     break
    return train_sentences, target

Below you will find the code to run the above functions, however I am using the pickle files from my submission to run this notebook.

In [27]:
# error_sents, error_target = prepare_data(combined_dev_test_pred)

In [28]:
# error_sents[:3]

## Working dataframe

Here, the working file for the counterfactual analysis is created with the necessary columns for this assignment.

I am loading the pickle files from my submission, therefore the code that saves the pickle files has been commented.

In [31]:
# with open("error_sents", 'wb') as f:
#     pickle.dump((error_sents, error_target), f)

with open("error_sents", 'rb') as f:
    loaded_data = pickle.load(f)

error_sents_loaded, error_target_loaded = loaded_data

In [32]:
error_sents_loaded[0]

[{'token': '``',
  'lemma': '``',
  'is_neg': False,
  'pos': 'PUNCT',
  'dep1_pos': 'VERB',
  'dep2_pos': 'ROOT',
  'common ancestor': 'S',
  'dep_path': 'punct ROOT neg',
  'neg_type': 'neg',
  'dep_path_length': 3,
  'dep': 'punct',
  'token_distance': 8,
  'same_chunk': False},
 {'token': 'Well',
  'lemma': 'well',
  'is_neg': False,
  'pos': 'INTJ',
  'dep1_pos': 'VERB',
  'dep2_pos': 'ROOT',
  'common ancestor': 'S',
  'dep_path': 'intj ROOT neg',
  'neg_type': 'neg',
  'dep_path_length': 3,
  'dep': 'intj',
  'token_distance': 7,
  'same_chunk': False},
 {'token': ',',
  'lemma': ',',
  'is_neg': False,
  'pos': 'PUNCT',
  'dep1_pos': 'VERB',
  'dep2_pos': 'ROOT',
  'common ancestor': 'S',
  'dep_path': 'punct ROOT neg',
  'neg_type': 'neg',
  'dep_path_length': 3,
  'dep': 'punct',
  'token_distance': 6,
  'same_chunk': False},
 {'token': 'Mrs.',
  'lemma': 'Mrs.',
  'is_neg': False,
  'pos': 'PROPN',
  'dep1_pos': 'PROPN',
  'dep2_pos': 'VERB',
  'common ancestor': 'S',
  'dep

In [33]:
len(error_sents_loaded)

437

In [34]:
input_file = 'combined_dev_test_pred.txt'

# flattening the list with token features
flattened_list = [
    [token_data['token'], token_data['lemma'], token_data['pos'], token_data['is_neg'], token_data['dep_path'], token_data['dep']]
    for sublist in error_sents_loaded for token_data in sublist
]

# reading data from input file
data = []
with open(input_file, mode='r', newline='', encoding='utf-8') as file:
    reader = csv.reader(file, delimiter='\t')  
    for row in reader:
        data.append(row)  # appending each row to the data list

# combining data with features
combined_data = []
for row, features in zip(data, flattened_list):
    combined_data.append(row + features)

# selecting specific columns
selected_columns = [0, 1, 2, 3, 8, -1, -2, -3, -4, -5, -7]
new_data = []

for row in combined_data:
    selected_row = [row[i] for i in selected_columns]
    new_data.append(selected_row)

# writing the final output 
counterfactual_file = 'counterfactual.tsv'
with open(counterfactual_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')
    for row in new_data:
        writer.writerow(row)



In [35]:
more_column_names = [
    "chapter_name",
    "sentence_number",
    "token_number",
    "word",
    "scope",
    "dep",
    "dep_path",
    "is_neg",
    "pos",
    "lemma",
    "pred",
]

final_df = pd.read_csv(counterfactual_file, sep="\t", header=None, names=more_column_names)

In [36]:
final_df

,chapter_name,sentence_number,token_number,word,scope,dep,dep_path,is_neg,pos,lemma,pred
0,circle01,0_1,0,``,_,punct,punct ROOT neg,False,PUNCT,``,_
1,circle01,0_1,1,Well,_,intj,intj ROOT neg,False,INTJ,well,_
2,circle01,0_1,2,",",_,punct,punct ROOT neg,False,PUNCT,",",_
3,circle01,0_1,3,Mrs.,_,compound,compound npadvmod ROOT neg,False,PROPN,Mrs.,_
4,circle01,0_1,4,Warren,_,npadvmod,npadvmod ROOT neg,False,PROPN,Warren,_
...,...,...,...,...,...,...,...,...,...,...,...
9297,wisteria02,436_3,16,propitiate,_,xcomp,xcomp dobj amod,False,VERB,propitiate,propitiate
9298,wisteria02,436_3,17,his,his,poss,poss dobj amod,False,PRON,his,his
9299,wisteria02,436_3,18,unclean,clean,amod,amod,True,ADJ,unclean,unclean
9300,wisteria02,436_3,19,gods,gods,dobj,dobj amod,False,NOUN,god,gods


## Counterfactual error analysis

### Adjectives following a copulative verb

**Conditions:**\
Adjective is part of the scope.\
Adjective appears in the predictions column.\
Accounted for adjectives containing negation cues.

In [40]:
columns_of_interest = ['chapter_name', 'sentence_number', 'word', 'dep', 'pred', 'scope']
df_filtered = final_df[columns_of_interest]

# grouping by 'chapter_name' and 'sentence_number'
grouped = final_df.groupby(['chapter_name', 'sentence_number'])

# checking if scope is in pred (for cases like 'regular' and 'irregular')
def is_scope_in_pred(scope, pred):
    return scope in pred or pred in scope

# current chapter and sentence
current_chapter = None
current_sentence = None
sentence_words = []
sentence_scope = []
sentence_pred = []

# counters for valid and invalid acomp sentences
counter_valid_acomp_sentences = 0
counter_invalid_acomp_sentences = 0

# going through each sentence
for (chapter, sentence_num), group in grouped:
    # filtering tokens with 'dep'='acomp' and 'scope'!="_" and checking if pred matches scope
    sentence_with_incorrect_predictions = [
        (row['word'], row['scope'], row['pred'])
        for _, row in group.iterrows()
        if row['dep'] == 'acomp' and row['scope'] != '_' and not is_scope_in_pred(row['scope'], row['pred'])
    ]
    
    sentence_with_valid_acomp = [
        (row['word'], row['scope'], row['pred'])
        for _, row in group.iterrows()
        if row['dep'] == 'acomp' and row['scope'] != '_' and is_scope_in_pred(row['scope'], row['pred'])
    ]

    # checking if there are any incorrect predictions (pred != scope)
    if sentence_with_incorrect_predictions:
        counter_invalid_acomp_sentences += 1

    if sentence_with_incorrect_predictions or sentence_with_valid_acomp:
        if chapter != current_chapter or sentence_num != current_sentence:
            # printing result for the previous sentence
            if sentence_words:
                print(f"\nSentence: Chapter: {current_chapter}, Sentence: {current_sentence}")
                print("Entire sentence: " + ' '.join(sentence_words))
                print("\nScope and Prediction differences:")
                print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
                print("-" * 45)
                for word, scope, pred in zip(sentence_words, sentence_scope, sentence_pred):
                    print(f"{word:<15} {scope:<15} {pred:<15}")
                print("\n") 

            # new sentence
            current_chapter = chapter
            current_sentence = sentence_num
            sentence_words = []  # empty list for new sentence
            sentence_scope = []
            sentence_pred = []

        # adding words to sentence_words, scope, and pred values
        for _, row in group.iterrows():
            sentence_words.append(row['word'])
            sentence_scope.append(row['scope'])
            sentence_pred.append(row['pred'])

        # printing words with dep=='acomp' and incorrect predictions
        if sentence_with_incorrect_predictions:
            print("\nIncorrect predictions:")
            for word, scope, pred in sentence_with_incorrect_predictions:
                print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
                print("-" * 45)
                print(f"{word:<15} {scope:<15} {pred:<15}")

        # printing words with dep=='acomp' and correct predictions
        if sentence_with_valid_acomp:
            print(f"\nCorrect predictions:")
            for word, scope, pred in sentence_with_valid_acomp:
                print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
                print("-" * 45)
                print(f"{word:<15} {scope:<15} {pred:<15}")

            # counter valid acomp sentences
            counter_valid_acomp_sentences += 1

# final sentence printing
if sentence_words:
    print(f"\nSentence: Chapter: {current_chapter}, Sentence: {current_sentence}")
    print("Entire sentence: " + ' '.join(sentence_words))
    print("\nScope and Prediction differences:")
    print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
    print("-" * 45)
    for word, scope, pred in zip(sentence_words, sentence_scope, sentence_pred):
        print(f"{word:<15} {scope:<15} {pred:<15}")
    print("\n")

# final count of valid and invalid acomp sentences
print(f"Number of sentences with acomp in scope and scope in pred: {counter_valid_acomp_sentences}")
print(f"Number of sentences with incorrect acomp predictions: {counter_invalid_acomp_sentences}")



Correct predictions:
Word            Scope           Pred           
---------------------------------------------
impossible      possible        impossible     

Sentence: Chapter: cardboard, Sentence: 1_1
Entire sentence: It is , however , unfortunately impossible entirely to separate the sensational from the criminal , and a chronicler is left in the dilemma that he must either sacrifice details which are essential to his statement and so give a false impression of the problem , or he must use matter which chance , and not choice , has provided him with .

Scope and Prediction differences:
Word            Scope           Pred           
---------------------------------------------
It              It              It             
is              is              is             
,               _               _              
however         _               _              
,               _               _              
unfortunately   _               unfortunately  
impossible      

### Secondary clause following a main clause with negation cue

**Conditions:**\
Negation cue in main clause.\
Root in scope.\
Sentence contains a subordinate clause in scope.\
Predictions lack root or subordinate clause.

In [43]:
# counters for correct and incorrect sentences
correct_counter = 0
incorrect_counter = 0
root_missing_counter = 0
subordinate_missing_counter = 0

for (chapter, sentence_num), group in grouped:
    # ccomp, advcl, relcl, and root rows
    ccomp_rows = group[(group['dep'] == 'ccomp')]
    advcl_rows = group[(group['dep'] == 'advcl')]
    relcl_rows = group[(group['dep'] == 'relcl')]  
    root_row = group[(group['dep'] == 'ROOT')]

    # at least one subordinate clause (ccomp/advcl/relcl) in scope
    subordinate_in_scope = any(row['scope'] != '_' for _, row in ccomp_rows.iterrows()) or \
                           any(row['scope'] != '_' for _, row in advcl_rows.iterrows()) or \
                           any(row['scope'] != '_' for _, row in relcl_rows.iterrows())
    # root_row is not empty 
    if root_row.empty:
        continue  # skip sentence if nothing in root
    # both root and subordinate are in scope for the sentence to be evaluated
    root_in_scope = root_row['scope'].iloc[0] != '_'
    if not root_in_scope or not subordinate_in_scope:
        continue

    # if ROOT is in pred
    root_in_pred = root_row['pred'].iloc[0] != '_'

    # if root is missing in pred, INCORRECT 1
    if not root_in_pred:
        incorrect_counter += 1
        root_missing_counter += 1
        print(f"\nMC_and_SC (Incorrect - Root missing in pred): Chapter: {chapter}, Sentence: {sentence_num}")
        print("Entire sentence: " + ' '.join(group['word']))
        print("\nDetails:")
        print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
        print("-" * 45)
        for _, row in group.iterrows():
            print(f"{row['word']:<15} {row['scope']:<15} {row['pred']:<15}")
        continue  

    # if subordinate clauses in scope are in pred
    subordinate_pred_missing = False
    for _, row in ccomp_rows.iterrows():
        if row['scope'] != '_' and row['pred'] == '_':
            subordinate_pred_missing = True
            break
    if not subordinate_pred_missing:
        for _, row in advcl_rows.iterrows():
            if row['scope'] != '_' and row['pred'] == '_':
                subordinate_pred_missing = True
                break
    if not subordinate_pred_missing:
        for _, row in relcl_rows.iterrows():
            if row['scope'] != '_' and row['pred'] == '_':
                subordinate_pred_missing = True
                break

    # If subordinate is missing in pred, INCORRECT 2
    if subordinate_pred_missing:
        incorrect_counter += 1
        subordinate_missing_counter += 1
        print(f"\nMC_and_SC (Incorrect - Subordinate missing in pred): Chapter: {chapter}, Sentence: {sentence_num}")
        print("Entire sentence: " + ' '.join(group['word']))
        print("\nDetails:")
        print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
        print("-" * 45)
        for _, row in group.iterrows():
            print(f"{row['word']:<15} {row['scope']:<15} {row['pred']:<15}")
        continue  

    # if no issues found, CORRECT
    correct_counter += 1
    print(f"\nMC_and_SC (Correct): Chapter: {chapter}, Sentence: {sentence_num}")
    print("Entire sentence: " + ' '.join(group['word']))
    print("\nDetails:")
    print(f"{'Word':<15} {'Scope':<15} {'Pred':<15}")
    print("-" * 45)
    for _, row in group.iterrows():  
        print(f"{row['word']:<15} {row['scope']:<15} {row['pred']:<15}")

# counters
print("\nSummary:")
print(f"Number of correct sentences: {correct_counter}")
print(f"Number of incorrect sentences: {incorrect_counter}")
print(f"Number of sentences where root is missing in pred: {root_missing_counter}")
print(f"Number of sentences where subordinate is missing in pred: {subordinate_missing_counter}")



MC_and_SC (Incorrect - Root missing in pred): Chapter: cardboard, Sentence: 156_1
Entire sentence: Again , carbolic or rectified spirits would be the preservatives which would suggest themselves to the medical mind , certainly not rough salt .

Details:
Word            Scope           Pred           
---------------------------------------------
Again           _               _              
,               _               _              
carbolic        _               _              
or              _               _              
rectified       _               _              
spirits         _               _              
would           would           _              
be              be              _              
the             the             _              
preservatives   preservatives   _              
which           which           _              
would           would           would          
suggest         suggest         suggest        
themselves      themselves 